# GP1 — linear-regime (SAOS) training, Table 3

Identifies the coefficients $c_1$ and $c_2$ of the linear form of eq. (7),
$\dot\gamma = c_1\sigma + c_2\dot\sigma$, at each sample age (2 to 702 min) of the
GP1 oscillation-time sweep ($\gamma_0$ = 0.05 %, $\omega$ = 10 rad/s).

The time sweep stores one stress amplitude and phase angle per age, so the stress waveform
is rebuilt at that age and a RhINN learns it while STRidge selects terms of the library

$$\dot\sigma = \lambda_1\sigma + \lambda_2\sigma^2 + \lambda_3\dot\gamma + \lambda_4\sigma\dot\sigma
+ \lambda_5\dot\sigma\sigma^2 + \lambda_6\dot\sigma\sigma^3 .$$

The result is read as a Maxwell model, $\dot\sigma = -\sigma/\lambda + G\dot\gamma$, so
$c_1 = (1/\lambda)/G$ and $c_2 = 1/G$ (Section 3). All training settings are in
`rhinn_sindy/settings.py`.

* **Output:** `results/retrained/GP1_SAOS.csv`, written after every age. Table 3 itself
  (`results/coefficients/GP1_SAOS.csv`) is never overwritten.
* **Runtime:** about 30 min per age on a laptop CPU.
* **Hyperparameters:** the settings in `rhinn_sindy/settings.py` are the ones used for the
  paper. Moving from one sample age to another changes the stress and strain magnitudes by
  orders of magnitude, so an age (or a new material) may need some hyperparameter tuning,
  mainly the loss weights, learning rate, number of rounds and STRidge tolerance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

from rhinn_sindy import figures
from rhinn_sindy.reproduce import available_ages, compare_with_paper, run_saos

MATERIAL = "GP1"
tf.random.set_seed(1234)
np.random.seed(1234)

## Choose the ages

Ages available for training (`age_min` as in Table 3):

In [ ]:
available_ages(MATERIAL, "SAOS")

In [ ]:
# "all" for the whole of Table 3, or a list of ages in minutes, e.g. [8, 702].
AGES = "all"

## Training data at the first and last age

In [ ]:
ages = available_ages(MATERIAL, "SAOS")["age_min"]
figures.saos_training_data(MATERIAL, [ages.iloc[0], ages.iloc[-1]])
plt.show()

## Train

In [ ]:
results = run_saos(MATERIAL, AGES)

## Compare with the fits behind Table 3

In [ ]:
compare_with_paper(MATERIAL, "SAOS")

## Check: integrate the identified Maxwell model

The trained $c_1, c_2$ are integrated with the imposed strain and compared with the
training waveform (up to four of the trained ages).

In [ ]:
figures.saos_fit(MATERIAL, results)
plt.show()